## 引用的包与一些基础数据

In [1]:
import pandas as pd
from src import charts, om_api
from src import info_id_my as im
import matplotlib

# 获取的天气指标
daily_indicators = im.id_my_daily_indicators

countries = im.countries


# 印尼 历史数据路径
id_path = ['data/weather/ID_2011-2020.csv',
           'data/weather/ID_2021-2024.csv',
           'data/weather/ID_202501-202508.csv']

# 马来 历史数据路径
my_path = ['data/weather/MY_2011-2020.csv',
           'data/weather/MY_2021-2024.csv',
           'data/weather/MY_202501-202508.csv']

countries[0]['csv_path'] = id_path
countries[1]['csv_path'] = my_path


[{'l0_code': 'ID',
  'l0_name': 'Indonesia',
  'l1_list': [{'l1_code': '1_RI',
    'l1_name': 'Riau',
    'title_tag': '(#1 20%)',
    'latitude': [1.55, -0.28, 0.59],
    'longitude': [100.73, 102.05, 100.98]},
   {'l1_code': '2_SU',
    'l1_name': 'North Sumatra',
    'title_tag': '(#2 12%)',
    'latitude': [3.92, 2.92, 1.48],
    'longitude': [98.18, 99.54, 99.94]},
   {'l1_code': '3_KT',
    'l1_name': 'Central Kalimantan',
    'title_tag': '(#3 12%)',
    'latitude': [-2.37, -2.38, -3.36],
    'longitude': [111.79, 112.74, 113.77]},
   {'l1_code': '4_KI',
    'l1_name': 'East Kalimantan',
    'title_tag': '(#4 10%)',
    'latitude': [-1.63, 0.19, 1.22],
    'longitude': [116.18, 116.9, 117.83]},
   {'l1_code': '5_KB',
    'l1_name': 'West Kalimantan',
    'title_tag': '(#5 9%)',
    'latitude': [1.48, 0.14, 0.26, -1.66],
    'longitude': [109.68, 110.44, 111.39, 110.46]},
   {'l1_code': '6_JA',
    'l1_name': 'Jambi',
    'title_tag': '(#6)',
    'latitude': [-0.94, -2.11, -1.83]

## 更新天气图表

In [2]:
api_start_date      = '2025-09-01'
api_end_date        = '2025-09-02'

forecast_start_date = '2025-09-02'
forecast_end_date   = '2025-09-17'


for contry in countries:

    ## 读取 CSV 历史数据（全部城市、历史已存档数据）
    csv_df = pd.DataFrame()
    for path in contry['csv_path']:
        df = pd.read_csv(path)
        csv_df = pd.concat([csv_df, df])
    csv_df['date'] = pd.to_datetime(csv_df['date'])
    csv_df = csv_df[ csv_df['date'] >= '2015-01-01' ]

    for city in contry['l1_list']:
        ## 读取 CSV 历史数据（指定城市、历史已存档数据）
        city_csv_df = csv_df[csv_df['l1_name'] == city['l1_name']]

        ## 请求 API 历史数据（指定城市、历史未存档数据）
        params = {
            'latitude'  : city['latitude'],
            'longitude' : city['longitude'],
            'start_date': api_start_date,
            'end_date'  : api_end_date,
            'daily_indicators': daily_indicators
        }
        api_history_df = om_api.daily_history(**params)
        api_history_df = api_history_df.groupby('date', as_index=False)[daily_indicators].mean()
        api_history_df['l1_name'] = city['l1_name']

        ## 请求 API 预测数据（指定城市、预测数据）
        params = {
            'latitude'  : city['latitude'],
            'longitude' : city['longitude'],
            'start_date': forecast_start_date,
            'end_date'  : forecast_end_date,
            'daily_indicators': daily_indicators
        }
        api_forecast_df = om_api.daily_forecast(**params)
        api_forecast_df = api_forecast_df.groupby('date', as_index=False)[daily_indicators].mean()
        api_forecast_df['l1_name'] = city['l1_name']

        ## 合并数据
        concat_df = pd.concat([city_csv_df, api_history_df, api_forecast_df])

        ## 数据处理
        concat_df = im.data_prapare(concat_df)

        ## 制图
        for style in im.id_my_styles:
            chart_params = {
                'min_history_year' : style['min_history_year'],
                'forecast_after' : forecast_start_date,
                'ylabel': style['ylabel'],
                'title' : style['title'] + city['l1_name'],
            }
            if 'ylim' in style:
                chart_params['ylim'] = style['ylim']
            chart = charts.day_annul_plot(concat_df, style['column'], **chart_params)
            path = f'./diagram/{contry['l0_code']}/{contry['l0_code']}{city['l1_code']}_{style['path']}.jpg'
            chart.savefig(path, dpi=300)
            matplotlib.pyplot.close()

## 合成大图

In [3]:
# 按照国家
for contry in countries:
    file_lists = []
    for city in contry['l1_list']:
        for style in im.id_my_styles:
            file_lists.append(f'diagram/{contry['l0_code']}/{contry['l0_code']}{city['l1_code']}_{style['path']}.jpg')
    charts.merge2grid(file_lists, len(contry['l1_list']), len(im.id_my_styles), f'diagram/grid/{contry['l0_code']}.jpg')


## 周报用图

In [5]:
s1 = [im.id_my_styles[0]['path'], #累计降水
      im.id_my_styles[1]['path'], #7日降水
      im.id_my_styles[3]['path'], #墒情
      im.id_my_styles[4]['path'], #温度
      im.id_my_styles[2]['path']] #30日降水
s2 = [im.id_my_styles[1]['path'], #7日降水
      im.id_my_styles[3]['path'],]#墒情
# 天气周报
file_lists = []
for contry in countries:
    for city in contry['l1_list']:
        for s in s1:
            file_lists.append(f'diagram/{contry['l0_code']}/{contry['l0_code']}{city['l1_code']}_{s}.jpg')
charts.merge2grid(file_lists[ 0:15], 3, len(s1), f'diagram/grid/wr1.jpg')
charts.merge2grid(file_lists[15:30], 3, len(s1), f'diagram/grid/wr2.jpg')
charts.merge2grid(file_lists[30:45], 3, len(s1), f'diagram/grid/wr3.jpg')
charts.merge2grid(file_lists[45:60], 3, len(s1), f'diagram/grid/wr4.jpg')

# 天气简报
file_lists = []
for contry in countries:
    for city in contry['l1_list']:
        for s in s2:
            file_lists.append(f'diagram/{contry['l0_code']}/{contry['l0_code']}{city['l1_code']}_{s}.jpg')
charts.merge2grid(file_lists[ 0:14], 7, len(s2), f'diagram/grid/wr5.jpg')
charts.merge2grid(file_lists[14:24], 5, len(s2), f'diagram/grid/wr6.jpg')

## 合并文件


In [5]:
# import pandas as pd
#
# a = pd.read_csv('data/weather/ID_202501-202507.csv', sep = ',')
# b = pd.read_csv('data/weather/ID_202508.csv', sep = ',')
# c = pd.concat([a, b])
#
# c.to_csv('data/weather/ID_202501-202508.csv', index=False)

## 保存历史数据

In [2]:
# import pandas as pd
# from src import om_api
#
# for country in countries:
#     all_df = pd.DataFrame()
#
#     for city  in country['l1_list']:
#         params = {
#             'latitude'  : city['latitude'],
#             'longitude' : city['longitude'],
#             'start_date': '2025-08-01',
#             'end_date'  : '2025-08-31',
#             'daily_indicators': daily_indicators
#         }
#         api_df = om_api.daily_history(**params)
#         groupby_df = api_df.groupby('date', as_index=False)[daily_indicators].mean()
#         groupby_df['l1_name'] = city['l1_name']
#         print(city['l1_name'])
#         all_df = pd.concat([all_df, groupby_df])
#
#     all_df.to_csv(f'data/weather/{country['l0_code']}_202508.csv', index=False)

Riau
North Sumatra
Central Kalimantan
East Kalimantan
West Kalimantan
Jambi
South Sumatra
Sabah
Sarawak
Johor
Pahang
Parak
